# Usage

## Available Cloud Physics Species

The cloud structures are based on the vapour pressures and nucleation rates of the cloud forming materials. The data and their sources are stored in nimbus/data/chem/cloud\_material.csv and can be adjusted to your need. If you have requests for further species, contact me (kiefersv.mail@gmail.com). Currently, the following materials are available:

<ul style="columns:4;">
  <li>Al2O3</li>
  <li>C</li>
  <li>CaTiO3</li>
  <li>CH4</li>
  <li>Cr</li>
  <li>CsCl</li>
  <li>Fe</li>
  <li>FeO</li>
  <li>H2O</li>
  <li>H2S</li>
  <li>KCl</li>
  <li>MgSiO3</li>
  <li>Mg2SiO4</li>
  <li>MnS</li>
  <li>NaCl</li>
  <li>Na2S</li>
  <li>NH3</li>
  <li>NH4Cl</li>
  <li>NH4SH</li>
  <li>S2</li>
  <li>S8</li>
  <li>SiO</li>
  <li>SiO2</li>
  <li>TiO2</li>
  <li>ZnS</li>
</ul>


## Available Opacity

Nimbus uses the opacity calculation of [Virga](https://github.com/natashabatalha/virga/tree/master) ([Batalha et al. 2026](https://iopscience.iop.org/article/10.3847/1538-3881/ae29e5)) and is compatible with their [refractive indices](https://zenodo.org/records/15886530) files. The standard installation of Nimbus already comes with the following species:

<ul style="columns:4;">
  <li>Al2O3</li>
  <li>CH4</li>
  <li>Cr</li>
  <li>Fe</li>
  <li>FeO</li>
  <li>H2O</li>
  <li>KCl</li>
  <li>MgSiO3</li>
  <li>Mg2SiO4</li>
  <li>MnS</li>
  <li>Na2S</li>
  <li>NH3</li>
  <li>SiO</li>
  <li>SiO2</li>
  <li>TiO2</li>
  <li>ZnS</li>
</ul>

## Main Nimbus functions

Nimbus comes with multiple setting options for the atmospheric structure and the numerical evaluation. Here we go quickly through each function, explaining the different settings available. The main object of Nimbus is defined via the following function which sets general evaluation parameters:

In [ ]:
nimb = Nimbus(working_dir='.', create_analytic_plots=False, verbose=False,
              mute=False)

- __working\_dir__: Directory to store all outputs and default search location for inputs.
- __create\_analytic\_plots__: If True, analytic plots of the intermediate outputs are created. This significantly slows down calculation but allows for additional insights and debugging.
- __verbose__: If True, additional information is printed. This affects runtime as a progress bar is enabled.
- __mute__: If True, Nimbus produces no print output at all.

The atmospheric structure is defined via the following function (which also performs all the structure calculations):


In [ ]:
nimb.set_up_atmosphere(temperature, pressure, kzz, mmw, gravity, species,
                       deep_mmr, fsed=1, metalicity=1, ignore_as_nucleator=[])

- __temperature__: Temperature structure of the atmosphere [K]
- __pressure__: Pressure structure of the atmosphere [bar]. Must have same dimension as temperature.
- __kzz__: Mixing strength coefficient of the atmosphere [cm$^2$/s]. Must have same dimension as temperature.
- __mmw__: Mean molecular weight of the atmosphere [amu]. Singular value for the whole atmosphere.
- __gravity__: Gravity of the planet [cm/s$^2$]. Singular value for the whole atmosphere.
- __species__: Name of the cloud material. Only one material can be given.
- __deep\_mmr__: Mass mixing ratio of the cloud below the cloud deck.
- __fsed__: Initial guess of the particle size, given as settling parameter. Default value is typically sufficient.
- __metallicity__: Gas-phase metallicity relative to solar used for the vapour pressure calculations of some cloud materials.
- __ignore_as_nucleator__: All species in this list will have their nucleation rate set to 0.

To simulate the cloud structure, the compute function is called. Here, general settings about how the cloud structure should be calculated can be set:

In [ ]:
nimb.compute(typ='convergence', rel_dif_in_mmr=1e-3, max_iterations=None,
             save_file=None, tag=None)

- __typ__: Determines the type of evaluation. Possibilities are
  - 'convergence': Iterates over fixed cloud particle radii until the maximum difference is below rel_dif_in_mmr.
  - 'iterate': Use a fixed number of iterations (particularly useful for retrival tasks)
  - 'full': Full evaluation of the underlying ODEs (no fixed radius assumption)
- __rel\_dif\_in\_mmr__: If type='convergence', the iteration will stop if the maximum difference is below this value.
- __max\_iterations__: If type='iterate', the iteration will stop after this number of repetitions.
- __save\_file__: If a string is given, the run will be saved under the given name in the working_dir.
- __tag__: Name under which the current run is saved internally. If None, 'last_run' is used.

## Output of Nimbus

The output of Nimbus is a xarray dataset with the following information saved. For iterative runs, the end state of each iteration is given in the 'all_' variables. For full runs, each time step is given instead.

| Variable                 | dimensions                     | Description                                      |
|--------------------------|--------------------------------|--------------------------------------------------|
| gas_mmr                  | (species, pressure)            | Gas-phase mass mixing ratio [g/g]                |
| cloud_mmr                | (species, pressure)            | Cloud material mass mixing ratio [g/g]           |
| nucleation_rate          | (species, pressure)            | Nucleation rates [1/cm$^3$/s]                    |
| growth_rate              | (species, pressure)            | Growth rates [1/cm$^3$/s]                        |
| cloud_number_density     | (pressure)                     | Number density of cloud particles [1/cm$^3$]     |
| gas_number_density       | (species, pressure)            | Number density of gas-phase particles [1/cm$^3$] |
| cloud_radius             | (pressure)                     | Radius of cloud particles [cm]                   |
| temperature              | (pressure)                     | Temperature structure [K]                        |
| rhoatmo                  | (pressure)                     | Atmospheric density [g/cm$^3$]                   |
| Kzz                      | (pressure)                     | Atmospheric mixing constant [cm$^2$/s]           |
| all_gas_mmr              | (species, time/iter, pressure) | Gas-phase mass mixing ratio [g/g]                |
| all_cloud_mmr            | (species, time/iter, pressure) | Cloud material mass mixing ratio [g/g]           |
| all_nucleation_rate      | (species, time/iter, pressure) | Nucleation rates [1/cm$^3$/s]                    |
| all_growth_rate          | (species, time/iter, pressure) | Growth rates [1/cm$^3$/s]                        |
| all_cloud_number_density | (time/iter,pressure)           | Number density of cloud particles [1/cm$^3$]     |
| all_gas_number_density   | (species, time/iter, pressure) | Number density of gas-phase particles [1/cm$^3$] |
| all_cloud_radius         | (time/iter, pressure)          | Radius of cloud particles [cm]                   |
| mmw                      | -                              | Mean molecular weight [g/mol]                    |
| total_iterations         | -                              | Number of iterations performed                   |
| tstart                   | -                              | Start time [s]                                   |
| tend                     | -                              | End time [s]                                     |
| tsteps                   | -                              | Number of timesteps in output                    |
| ode_rtol                 | -                              | Relative tolarance of ODE solver                 |
| ode_atol                 | -                              | Absolute tolarance of ODE solver                 |
| ode_minimum_mmr          | -                              | Minimum mmr for ODE solver                       |
| static_rg                | -                              | True for iterative approache                     |
| r_ccn                    | -                              | CCN radius [cm]                                  |
| cs_mol                   | -                              | molecular cross section                          |
| eps_k                    | -                              | Depth of the Lennard-Jones potential             |
| rg_fit_deg               | -                              | Radius fitting degree for iterative runs         |

## Spectra calculations

To generate spectra from cloud models, Nimbus uses Virga routines to connect to PICASO. To create opacities and store them in a format that can be read into PICASO, use the following function:

In [ ]:
df = nimb.picaso_formater(tag='last_run', path_to_opacities=None, sig=2,
                          mie_type='full', nradii=100)

- __tag__: Tag of the run to be calculated. By default it is 'last_run'.
- __path_to_opacities__: Which opacity files to use. Default are the Nimbus ones.
- __sig__: Standard deviation of the lognormal size distribution used to calculate opacities.
- __mie_type__: Type of mie calculation, currently available
  - 'full': Use full mie calculation within MieAi.
  - 'grid': Use grid interpolation of MieAi (requires a suitable grid)
- __nradii__: Number of radius bins for calculation.

## Additional Nimbus Settings

In addition to the main settings, these functions allow for more detailed control over the Nimbus run. In general, it is not expected that you will need to adjust any of these variables. The following function can be used to adjust settings for the ODE solver:

In [ ]:
 nimb.set_solver_settings(
     initial_time_for_solver=None, end_time_for_solver=None,
     evaluation_steps_for_solver=None, degree_of_radius_polinomial=None,
     rtol=None, atol=None, ode_minimum_mmr=None
 )

- __initial\_time\_for\_solver__: This sets the start time of the solver (only relevant for the diagnostic output)
- __end\_time\_for\_solver__: The maximum evaluation time of the solver. It is important to set this valeu large enough for the cloud model to be able to reach a static solution. Default value is 1e15 seconds.
- __evaluation\_steps\_for\_solver__: Number of evaluation points (only relevant for the diagnostic output)
- __degree\_of\_radius\_polinomial__: Degree of the polynomial used for radius smoothing. Default is 8.
- __rtol__: Relative tolarance of the solver. Default is 1e-6.
- __atol__: Absolute tolerance of the solver. Default is 1e-25.
- __ode\_minimum\_mmr__: Lower limit for the solver, everything lower is considered to be 0 (should be lower than atol).

The cloud physics parameters can be set via the following function:

In [ ]:
nimb.set_cloud_settings(minimum_cloud_particle_radius=None, molecular_cross_section=None)

- __minimum\_cloud\_particle\_radius__: Lower size limit of the cloud particles, considered to be the CCN radius. Default is 1e-7 cm.
- __molecular\_cross\_section__: Molecular cross-section. Default is set for an H$_2$ gas and is 2e-15 cm$^2$.

In addition, you might want to add some additional factors to certain parts of the equations. Some of these tweaks (or fudges) are predefined. The default for all values is 1.

In [ ]:
nimb.set_fudge_settings(
    nucleation_rate_fudge=None, accreation_rate_fudge=None,
    sticking_coefficient=None
)

- __nucleation\_rate\_fudge__: Multiplicative factor to the nucleation rate to study how uncertainties in the nucleation rate affect the cloud structure.
- __accreation\_rate\_fudge__: Decrypted, same as sticking\_coefficient.
- __sticking\_coefficient__: Chance of a collisional limited accretion reaction to result in the growth of the cloud particle.

## DataBase

The DataBase class is an internal functionality that handles the data used by Nimbus. You can also use this class to access the data of Nimbus.

In [ ]:
from nimbus import DataBase

ds = DataBase(data_file=None)

The datafile can be used to give a self written database. The default file is located at nimbus/data/chem/cloud\_material.csv. The following (self explenatory) functions are available:

In [ ]:
m_s = ds.monomer_mass(species)  # [g]
R_spec_s = ds.specific_gas_constant(species)  # [erg/g/K]
r_s = ds.monomer_radius(species)  # [cm]
mu_s = ds.molecular_weight(species)  # [amu]
rho_s = ds.solid_density(species)  # [g/cm3]
sigam_s = ds.surface_tension(species, temp)  # [erg/cm2]
G_s = ds.gibbs_free_energy(species, temp)  # [erg]
p_vap_s = ds.vapor_pressures(species, temp, metalicity=1)  # [dyn/cm$^2$]